In [ ]:
%pip install bert-score

import pandas as pd
import torch
from transformers.models.auto.processing_auto  import AutoProcessor
from transformers.models.auto.modeling_auto import AutoModelForVisualQuestionAnswering
from PIL import Image
import os
from sklearn.metrics import accuracy_score, f1_score
from bert_score import score as bert_score
from tqdm import tqdm
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing in

2025-05-17 20:21:23.729508: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747513284.147697      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747513284.268872      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [10]:
# Define paths to dataset
dataset_root = "/kaggle/input/dataset"
vqa_csv_path = os.path.join(dataset_root, "vqa_dataset_final_new.csv")
image_metadata_path = os.path.join(dataset_root, "images.csv")
image_dir = os.path.join(dataset_root, "small/small")
output_path = "/kaggle/working/blip-vqa-base_summary.csv"

In [ ]:
# Step 1: Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Step 2: Load BLIP-VQA-Base model and processor
processor = AutoProcessor.from_pretrained("Salesforce/blip-vqa-base",useFast=True)
# Step 2: Load BLIP-VQA-Base model and processor
processor = AutoProcessor.from_pretrained("Salesforce/blip-vqa-base")
model = AutoModelForVisualQuestionAnswering.from_pretrained(
    "Salesforce/blip-vqa-base", torch_dtype=torch.float16
)
model.to(device)
model.eval()

Using device: cuda


BlipForQuestionAnswering(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-05, e

In [12]:
# Step 3: Load VQA dataset and image metadata
vqa_data = pd.read_csv(vqa_csv_path)
print(f"Loaded {len(vqa_data)} VQA entries")

image_metadata = pd.read_csv(image_metadata_path)[["image_id", "path"]]
print(f"Loaded {len(image_metadata)} image metadata entries")

# Verify required columns in VQA dataset
required_vqa_columns = ["image_id", "question", "answer"]
if not all(col in vqa_data.columns for col in required_vqa_columns):
    print(f"Error: VQA CSV must contain {required_vqa_columns}")
    exit(1)

# Step 4: Merge VQA data with image metadata to get image paths
vqa_data = vqa_data.merge(image_metadata, on="image_id", how="inner")
vqa_data["image_path"] = vqa_data["path"].apply(lambda x: os.path.join(image_dir, x))
vqa_data = vqa_data[vqa_data["image_path"].apply(os.path.exists)]

Loaded 21888 VQA entries
Loaded 398212 image metadata entries


In [13]:
# Step 5: Evaluate model on entire dataset
predictions = []
ground_truths = []
valid_indices = []

for idx, row in tqdm(vqa_data.iterrows(), total=len(vqa_data), desc="Evaluating"):
    image_path = row["image_path"]
    question = f"{row['question']}. Return a single word."
    ground_truth = str(row["answer"]).strip().lower().split()[0]

    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text=question, return_tensors="pt").to(device, torch.float16)

    # Generate answer with beam search
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            num_beams=5,
            no_repeat_ngram_size=2
        )
    predicted_answer = processor.decode(outputs[0], skip_special_tokens=True).strip().lower()

    # Post-process prediction
    predicted_answer = re.sub(r"[^\w\s]|'s|\?s", "", predicted_answer)
    predicted_answer = predicted_answer.split()[0] if predicted_answer.split() else "unknown"

    predictions.append(predicted_answer)
    ground_truths.append(ground_truth)
    valid_indices.append(idx)

Evaluating: 100%|██████████| 21888/21888 [33:56<00:00, 10.75it/s]


In [14]:
# Step 6: Compute evaluation metrics
# Accuracy
accuracy = accuracy_score(ground_truths, predictions)
print(f"Accuracy: {accuracy:.4f}")

# F1 Score (macro)
f1 = f1_score(ground_truths, predictions, average="macro")
print(f"F1 Score (Macro): {f1:.4f}")

# BERTScore
P, R, F1 = bert_score(predictions, ground_truths, lang="en", verbose=True)
bertscore_f1 = F1.mean().item()
print(f"BERTScore F1: {bertscore_f1:.4f}")

Accuracy: 0.3232
F1 Score (Macro): 0.0881


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/39 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/342 [00:00<?, ?it/s]

done in 7.34 seconds, 2983.81 sentences/sec
BERTScore F1: 0.9628


In [15]:
# Step 7: Save results to CSV
results_df = vqa_data.iloc[valid_indices].copy()
results_df["predicted_answer"] = predictions
results_df["is_correct"] = [pred == gt for pred, gt in zip(predictions, ground_truths)]
results_df[["image_id", "question", "answer", "predicted_answer", "is_correct"]].to_csv(output_path, index=False)
print(f"Saved predictions and metrics to {output_path}")

# Step 8: Save metrics summary to CSV
metrics = {
    "accuracy": accuracy,
    "f1_score_macro": f1,
    "bertscore_f1": bertscore_f1,
    "num_evaluated": len(predictions)
}
metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv("/kaggle/working/blip-vqa-base_metrics.csv", index=False)
print("\nMetrics Summary:")
print(metrics_df)

Saved predictions and metrics to /kaggle/working/blip-vqa-base_summary.csv

Metrics Summary:
   accuracy  f1_score_macro  bertscore_f1  num_evaluated
0  0.323236        0.088085       0.96276          21888
